In [1]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import sys

if ".." not in sys.path:
    sys.path.append("..")

from model.efficientnet import create_efficientnet_b3

print("Model import successful!")

Model import successful!


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [4]:
model = create_efficientnet_b3(
    num_classes=5,
    pretrained=True
)

model = model.to(device)

print("EfficientNetB3 created successfully!")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to C:\Users\91828/.cache\torch\hub\checkpoints\efficientnet_b3_rwightman-b3899882.pth


100.0%


EfficientNetB3 created successfully!


In [5]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1536, out_features=5, bias=True)
)


In [6]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 10,703,917
Trainable parameters: 10,703,917


In [7]:
import pandas as pd

from model.datasets import DRDataset
from preprocessing.augment import get_val_transform

val_df = pd.read_csv(
    "../dataset/splits/val_split.csv"
)

val_dataset = DRDataset(
    dataframe=val_df,
    image_dir="../dataset/train_images",
    transform=get_val_transform()
)

print("Validation samples:", len(val_dataset))

Validation samples: 366


In [8]:
from torch.utils.data import DataLoader

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0
)

images, labels = next(iter(val_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([4, 3, 300, 300])
Labels: torch.Size([4])


In [9]:
images = images.to(device)
labels = labels.to(device)

print("Images device:", images.device)
print("Labels device:", labels.device)

Images device: cuda:0
Labels device: cuda:0


In [10]:
model.eval()

with torch.no_grad():
    outputs = model(images)

print("Output shape:", outputs.shape)
print(outputs)


Output shape: torch.Size([4, 5])
tensor([[ 0.0663,  0.1313,  0.1484,  0.1973, -0.0799],
        [ 0.0948,  0.1296,  0.0458,  0.3195, -0.0833],
        [ 0.1900,  0.3494,  0.2719, -0.0477, -0.2201],
        [ 0.0047,  0.1049,  0.0936,  0.1687, -0.1088]], device='cuda:0')


In [11]:
probabilities = torch.softmax(
    outputs,
    dim=1
)

print(probabilities)

tensor([[0.1939, 0.2070, 0.2105, 0.2211, 0.1676],
        [0.1970, 0.2040, 0.1876, 0.2466, 0.1649],
        [0.2123, 0.2490, 0.2304, 0.1674, 0.1409],
        [0.1898, 0.2098, 0.2074, 0.2236, 0.1694]], device='cuda:0')


In [12]:
print(
    probabilities.sum(dim=1)
)

tensor([1.0000, 1.0000, 1.0000, 1.0000], device='cuda:0')


In [13]:
predictions = torch.argmax(
    probabilities,
    dim=1
)

print("Predictions:", predictions)
print("Actual labels:", labels)

Predictions: tensor([3, 3, 1, 3], device='cuda:0')
Actual labels: tensor([0, 0, 2, 0], device='cuda:0')


In [14]:
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

In [15]:
for i in range(len(predictions)):

    predicted_class = predictions[i].item()
    actual_class = labels[i].item()

    confidence = probabilities[i][
        predicted_class
    ].item()

    print(
        f"Image {i+1}: "
        f"Predicted = {class_names[predicted_class]}, "
        f"Actual = {class_names[actual_class]}, "
        f"Confidence = {confidence:.2%}"
    )

Image 1: Predicted = Severe, Actual = No DR, Confidence = 22.11%
Image 2: Predicted = Severe, Actual = No DR, Confidence = 24.66%
Image 3: Predicted = Mild, Actual = Moderate, Confidence = 24.90%
Image 4: Predicted = Severe, Actual = No DR, Confidence = 22.36%
